In [2]:
from osgeo import ogr, osr
import pandas as  pd

fn = r"C:\Users\TheOne\Dataware\GeoPy\cities.csv"
df = pd.read_csv(fn)
print(df)

    rank         name  population  pop_density   country   latitude  \
0      1     Shanghai    24150000         3809     China  31.224146   
1      2      Karachi    23500000         6663  Pakistan  24.926710   
2      3      Beijing    21516000         1311     China  39.906570   
3      4      Tianjin    14722100         3647     China  39.144939   
4      5     Istanbul    14377019         2633    Turkey  41.060730   
..   ...          ...         ...          ...       ...        ...   
79    80      Nairobi     3138369         4829     Kenya  -1.273670   
80    81    Zhongshan     3121275         1750     China  22.519051   
81    82       Ürümqi     3112559          214     China  43.787869   
82    83  Addis Ababa     3103673         5889  Ethiopia   9.033600   
83    84      Wenzhou     3039439         2559     China  28.010321   

     longitude  
0   121.516136  
1    67.034370  
2   116.387649  
3   117.200569  
4    28.987770  
..         ...  
79   36.874901  
80  113.367

In [3]:
# 创建DataSource
driver: ogr.Driver = ogr.GetDriverByName('ESRI Shapefile')
ds: ogr.DataSource = driver.CreateDataSource(r"C:\Users\TheOne\Dataware\GeoPy\cities.shp")

# 创建WGS84空间参考
srs = osr.SpatialReference()
srs.ImportFromEPSG(4326)

# 创建图层
layer: ogr.Layer = ds.CreateLayer('Cities', srs, ogr.wkbPoint)
# 添加属性定义
name = ogr.FieldDefn('Name', ogr.OFTString)
name.SetWidth(24)
layer.CreateField(name)
population = ogr.FieldDefn('Population', ogr.OFTReal)
layer.CreateField(population)
country = ogr.FieldDefn('Country', ogr.OFTString)
country.SetWidth(24)
layer.CreateField(country)

# 新建Feature并且给其属性赋值
def add_feature(item):
    feature = ogr.Feature(layer.GetLayerDefn())
    feature.SetField('Name', item['name'])
    feature.SetField('Population', item['population'])
    feature.SetField('Country', item['country'])

    # 设置Feature的几何属性Geometry
    point = ogr.CreateGeometryFromWkt(f"POINT ({item['longitude']} {item['latitude']})")
    feature.SetGeometry(point)
    # 创建Feature
    layer.CreateFeature(feature)

# 使用apply函数将表格中每行信息转为要素，添加到图层，axis=1表示将add_feature()函数作用于DataFrame的行
df.apply(add_feature, axis=1)
ds.FlushCache()

del ds

In [4]:
from osgeo import ogr
ogr.UseExceptions()


# 首先定义每个省全称到简称的映射字典
names = {
    '北京': '京',
    '天津': '津',
    '重庆': '渝',
    '上海': '沪',
    '河北': '冀',
    '山西': '晋',
    '辽宁': '辽',
    '吉林': '吉',
    '黑龙江': '黑',
    '江苏': '苏',
    '浙江': '浙',
    '安徽': '皖',
    '福建': '闽',
    '江西': '赣',
    '山东': '鲁',
    '河南': '豫',
    '湖北': '鄂',
    '湖南': '湘',
    '广东': '粤',
    '海南': '琼',
    '四川': '川/蜀',
    '贵州': '黔/贵',
    '云南': '云/滇',
    '陕西': '陕/秦',
    '甘肃': '甘/陇',
    '青海': '青',
    '台湾': '台',
    '内蒙古': '蒙',
    '广西': '桂',
    '宁夏': '宁',
    '新疆': '新',
    '西藏': '藏',
    '香港': '港',
    '澳门': '澳'
}

# 打开一个Shapefile文件
ds: ogr.DataSource = ogr.Open(r"C:\Users\TheOne\Dataware\GeoPy\china-province\Province.shp", update=True)
layer: ogr.Layer = ds.GetLayer()

# 添加一个省简称的字段
field: ogr.FieldDefn = ogr.FieldDefn('Abbr', ogr.OFTString)
field.SetWidth(5)
layer.CreateField(field)

# 填充属性值
for feature in layer:
    name: str = feature.GetField('NAME')
    feature.SetField('Abbr', names.get(name, ''))
    # 修改完了记得Set一下
    layer.SetFeature(feature)

# 关闭数据集
del ds

In [2]:
from osgeo import ogr
ogr.UseExceptions()


# 从给定图层中读取字段的定义，根据给定字段名称找到该字段的索引编号
def get_field_index_by_name(layer: ogr.Layer, name: str):
    defn: ogr.FeatureDefn = layer.GetLayerDefn()
    for i in range(defn.GetFieldCount()):
        if name == defn.GetFieldDefn(i).GetName():
            return i
    # 如果没有找到对应的字段，则抛出异常
    raise ValueError(f'{name} not found')


# 打开一个Shapefile文件
ds: ogr.DataSource = ogr.Open(r"C:\Users\TheOne\Dataware\GeoPy\china-province\Province.shp", update=True)
layer: ogr.Layer = ds.GetLayer()
# 删除Abbr字段
index = get_field_index_by_name(layer, 'Abbr')
layer.DeleteField(index)
del ds

In [ ]:
from osgeo import ogr
ogr.UseExceptions()


# 打开一个Shapefile
ds: ogr.DataSource = ogr.Open(r"C:\Users\TheOne\Dataware\GeoPy\china-province\Province.shp", update=True)
layer: ogr.Layer = ds.GetLayer()

# 填充属性值
for feature in layer:
    name: str = feature.GetField('NAME')
    if name in ('北京', '天津', '重庆', '上海'):
        name += '市'
    elif name in ('内蒙古', '广西', '宁夏', '新疆', '西藏'):
        name += '自治区'
    elif name in ('香港', '澳门'):
        name += '特别行政区'
    else:
        name += '省'
    feature.SetField('NAME', name)
    # 修改完了记得Set一下
    layer.SetFeature(feature)

# 关闭数据集
del ds

In [2]:
from osgeo import ogr
ogr.UseExceptions()

fn: str = r"C:\Users\TheOne\Dataware\GeoPy\china-province\Province.shp"
ds: ogr.DataSource = ogr.Open(fn)
# 注意Layer的名称中尽量不要包含中文
layer: ogr.Layer = ds.GetLayer()
# 选择出中学数量大于1万所的省份，这里的NAME和HighSchool为属性名称
query: str = f'SELECT NAME, HighSchool FROM {layer.GetName()} WHERE HighSchool > 10000'
selected: ogr.Layer = ds.ExecuteSQL(query)
# 这里的Feature中只包含两个属性NAME和HighSchool
for feature in selected:
    print(feature.GetField('NAME'))

# 选择出中学数量最多的省份
query: str = f'SELECT NAME, HighSchool FROM {layer.GetName()} ORDER BY HighSchool DESC'
selected: ogr.Layer = ds.ExecuteSQL(query)
print(f"中学数量最多的省份（地区）：{selected.GetFeature(0).GetField('NAME')}")
print(f"中学数量最多的省份（地区）具有{int(selected.GetFeature(0).GetField('HighSchool'))}所学校")

黑龙江
新疆
海南
吉林
辽宁
天津
内蒙古
上海
北京
中学数量最多的省份（地区）：上海
中学数量最多的省份（地区）具有19532所学校


In [3]:
from osgeo import ogr
ogr.UseExceptions()

fn: str = r"C:\Users\TheOne\Dataware\GeoPy\china-province\Province.shp"
ds: ogr.DataSource = ogr.Open(fn)
# 注意Layer的名称不能包含中文
layer: ogr.Layer = ds.GetLayer()
# 使用filter函数对要素属性进行过滤
selected = list(filter(lambda f: f.GetField('HighSchool') > 10000, layer))
for feature in selected:
    print(feature.GetField('NAME'))

# 使用sorted方法对要素进行自定义排序，这里使用逆序
selected = sorted(layer, key=lambda f: f.GetField('HighSchool'), reverse=True)
print(selected[0].GetField('NAME'))
print(selected[0].GetField('HighSchool'))

黑龙江
新疆
海南
吉林
辽宁
天津
内蒙古
上海
北京
上海
19532.0
